
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lecture - Production Evaluation and Monitoring
#### Duration: 30 minutes

## Overview
This lecture covers the full spectrum of production evaluation and monitoring for generative AI applications on Databricks. It begins with the four scorer types available for automated quality assessment, progresses through multi-turn conversation judges for session-level evaluation, and then covers sampling strategies, the scorer lifecycle, management APIs, metric backfill for historical traces, and storing traces in Unity Catalog for long-term analysis.

## Learning Objectives
By the end of this lecture you will be able to:
- Differentiate the four scorer types and select the appropriate one for a given evaluation need
- Construct serializable custom scorers using the `@scorer` decorator
- Configure multi-turn judges for conversation-level quality assessment
- Design a sampling strategy that balances safety coverage, quality signal, and cost
- Manage scorers through their full lifecycle: register, start, update, stop, and delete
- Apply metric backfill to evaluate historical traces with new scorers
- Explain why Databricks recommends storing production traces in Unity Catalog, and how to query them with SQL

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Prerequisites</strong>
<p style="margin: 8px 0 0 0; color: #333;">This lecture assumes familiarity with MLflow Tracing and agent deployment, including how traces are logged to MLflow Experiments.</p>
</div>
</div>
</div>

## A. Scorer Types
In MLflow, a **scorer** is the evaluation function you use to assess model or agent outputs, while a **judge** is a specific type of scorer that uses a Large Language Model (LLM) to perform the evaluation. Judges are LLM-powered and focus on subjective or nuanced criteria, whereas scorers can also be deterministic code-based functions for objective checks. Judges must be wrapped in scorers to be used in the evaluation harness or production monitoring. You can read more about the nuances between scorers and judges here ([AWS](https://docs.databricks.com/aws/en/mlflow3/genai/concepts/#scorers) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/concepts/) | [GCP](https://docs.databricks.com/gcp/en/mlflow3/genai/concepts/)).

### A1. Built-in LLM Judges

MLflow provides **pre-built LLM judges** for common quality dimensions. These are ready-to-use scorers that require no custom prompt engineering. Each judge uses a Databricks-hosted LLM by default, or you can set a Unity Gateway model service with `model="databricks:/&lt;catalog&gt;.&lt;schema&gt;.&lt;model-service&gt;"`.

The standard pattern for using any built-in judge is two steps: **register** (bind to an experiment) and **start** (begin evaluating traces):

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: Registering and starting a Safety judge</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">from mlflow.genai.scorers import Safety, ScorerSamplingConfig
<span></span>
# 1. Register the scorer (binds to active experiment)
safety_judge = Safety(
    model="databricks:/workspace.agent_deployment.judge-model"
).register(name="my_safety_judge")
<span></span>
# 2. Start with a sampling rate (safety checks run on every trace)
safety_judge = safety_judge.start(
    sampling_config=ScorerSamplingConfig(sample_rate=1.0)
)</div>
</div>
</details>

The built-in judges span safety, response quality, RAG, and tool calls. Guidelines judges are covered in A2.

| Category | Judges | Ground truth? |
|---|---|---|
| **Safety** | `Safety` | No |
| **Response quality** | `RelevanceToQuery` | No |
| | `Correctness` | Yes |
| **RAG** | `RetrievalRelevance`, `RetrievalGroundedness` | No |
| | `RetrievalSufficiency` | Yes |
| **Tool calls** | `ToolCallEfficiency` | No |
| | `ToolCallCorrectness` | Optional |

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">All built-in judges accept a <code>model=</code> parameter that sets the judge LLM. Point it at a Unity Gateway model service: <code>"databricks:/&amp;lt;catalog&amp;gt;.&amp;lt;schema&amp;gt;.&amp;lt;model-service&amp;gt;"</code>. This course uses <code>workspace.agent_deployment.judge-model</code>. The <code>.register(name=...)</code> name must be unique within the experiment.</p>
</div>
</div>
</div>

Pre-built judges and scorers (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/concepts/scorers" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/concepts/scorers" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/eval-monitor/concepts/scorers" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

### A2. Guidelines Judges

**Guidelines judges** evaluate traces against natural-language criteria you define. Each guideline is a pass/fail rule. The judge returns `Feedback` with value `"yes"` or `"no"` (strings, not booleans).

- Takes a `name` and a `guidelines` list of plain-English rules
- Guidelines can reference **context variable keys** by name (e.g., `retrieved_documents`, `max_allowed_discount`)
- `ExpectationsGuidelines` is the ground-truth variant: it reads per-row guidelines from each example's `expectations`, so it needs labeled data
- Set the judge model with `model="databricks:/&lt;catalog&gt;.&lt;schema&gt;.&lt;model-service&gt;"`

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: Guidelines judge with custom rules</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">from mlflow.genai.scorers import Guidelines, ScorerSamplingConfig
<span></span>
english_judge = Guidelines(
    name="english",
    guidelines=["The response must be in English"],
    model="databricks:/workspace.agent_deployment.judge-model",
).register(name="is_english")
<span></span>
english_judge = english_judge.start(
    sampling_config=ScorerSamplingConfig(sample_rate=0.5)
)</div>
</div>
</details>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">Guidelines judges are the fastest way to encode domain-specific quality rules without writing code. Write the rule in plain English and the LLM judge evaluates each trace against it.</p>
</div>
</div>
</div>

Guidelines judges and scorer types (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/concepts/scorers" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/concepts/scorers" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/eval-monitor/concepts/scorers" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

### A3. Custom Prompt Judges

Use `make_judge` from `mlflow.genai.judges` when you need **full control over the evaluation prompt**. This lets you write a custom template with specific instructions and define multi-level quality scales.

- Only reserved template variables are allowed; custom variable names throw validation errors
- Single-turn judges use `{{inputs}}`, `{{outputs}}`, `{{expectations}}`, and `{{trace}}`
- Session-level (multi-turn) judges use `{{conversation}}` instead, which can be combined only with `{{expectations}}`
- Specify quality levels with `feedback_value_type` using `Literal` types
- The `model` parameter is required when using `{{trace}}`-based judges; optional otherwise

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: Custom prompt judge with make_judge()</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">from typing import Literal
from mlflow.genai.judges import make_judge
<span></span>
formality_judge = make_judge(
    name="formality",
    instructions="""Evaluate whether the response is
      formal, somewhat formal, or not formal.
      Request: {{inputs}}
      Response: {{outputs}}""",
    feedback_value_type=Literal[
        "formal", "semi_formal", "not_formal"
    ],
    model="databricks:/workspace.agent_deployment.judge-model",
).register(name="formality_judge")</div>
</div>
</details>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">Custom prompt judges sit between guidelines judges (natural-language rules) and custom function scorers (arbitrary Python). Use them when you need a specific prompt structure or multi-level quality scales that go beyond pass/fail.</p>
</div>
</div>
</div>


### A4. Custom Function Scorers

The `@scorer` decorator turns any Python function into an MLflow-compatible evaluation function. This gives you **full control**: regex checks, schema validation, keyword matching, business rules: anything you can express in Python.

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: @scorer decorator with aggregations</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">from mlflow.genai.scorers import scorer
<span></span>
@scorer
def mentions_databricks(outputs):
    return "databricks" in str(outputs.get("response", "")).lower()
<span></span>
@scorer(aggregations=["mean", "min", "max"])
def response_length(outputs):
    return len(str(outputs.get("response", "")))</div>
</div>
</details>

**Serialization rules:** custom scorers run remotely and must be fully self-contained:

| Rule | Details |
|---|---|
| `import` inline | Every import must be inside the function body |
| Notebook-defined | Must be defined in a Databricks notebook |
| No class-based scorers | `Scorer` subclasses cannot be serialized for remote execution |
| No external references | Cannot reference variables or modules defined outside the function |
| No import-requiring type hints | Signature hints cannot reference types that need imports |

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">Use <code>aggregations=["mean", "min", "max"]</code> on numeric scorers to get experiment-level summary statistics. Aggregated metrics appear in the experiment view for trend analysis.</p>
</div>
</div>
</div>

Custom scorers with @scorer (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-scorers" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/custom-scorers" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/eval-monitor/custom-scorers" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

## B. Multi-Turn Judges

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Experimental</strong>
<p style="margin: 8px 0 0 0; color: #333;">Multi-turn evaluation is Experimental. Its API and behavior might change in future releases.</p>
</div>
</div>
</div>

### B1. Conversation Judges

Multi-turn judges evaluate **entire conversations**, not individual requests. They operate on sessions (groups of traces that share the same `mlflow.trace.session` metadata value). Assessments attach to the **first trace** in the session.

<div style="background: #fff; max-width: 1600px; margin: 0 auto; padding: 12px 0 24px;">
<svg viewBox="0 0 1600 482" width="100%" role="img" aria-label="A three-turn session for user_303, who asks three times to cancel a hotel reservation. Safety assesses each trace. The multi-turn judges UserFrustration and ConversationCompleteness assess the whole session and attach their results to Trace 1." style="display: block; height: auto; font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;">
<defs>
<marker id="sj-ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
<filter id="sj-sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="3" stdDeviation="6" flood-color="#1B3139" flood-opacity="0.08"/></filter>
</defs>
<rect x="40" y="20" width="1520" height="378" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#sj-sh)"/>
<rect x="62" y="34" width="1476" height="5" rx="2.5" fill="#1B5162"/>
<rect x="62" y="53" width="188" height="26" rx="13.0" fill="#1B5162"/><text x="156" y="71" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">SESSION · user_303</text>
<rect x="62" y="94" width="1476" height="82" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<text x="86" y="144" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Trace 1</text>
<text x="210" y="144" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">“I need to cancel my hotel reservation in Barcelona.”</text>
<rect x="900" y="118" width="85" height="34" rx="17.0" fill="#EAF2FA" stroke="#2272B4" stroke-width="1.2"/><text x="943" y="141" font-size="17" font-weight="700" fill="#2272B4" text-anchor="middle" xml:space="preserve">Safety</text>
<rect x="997" y="118" width="171" height="34" rx="17.0" fill="#FFEFEC" stroke="#FF5F46" stroke-width="1.2"/><text x="1083" y="141" font-size="17" font-weight="700" fill="#FF5F46" text-anchor="middle" xml:space="preserve">UserFrustration</text>
<rect x="1180" y="118" width="256" height="34" rx="17.0" fill="#FFEFEC" stroke="#FF5F46" stroke-width="1.2"/><text x="1308" y="141" font-size="17" font-weight="700" fill="#FF5F46" text-anchor="middle" xml:space="preserve">ConversationCompleteness</text>
<rect x="62" y="190" width="1476" height="82" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<text x="86" y="240" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Trace 2</text>
<text x="210" y="240" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">“No, I said I need to CANCEL my reservation.”</text>
<rect x="900" y="214" width="85" height="34" rx="17.0" fill="#EAF2FA" stroke="#2272B4" stroke-width="1.2"/><text x="943" y="237" font-size="17" font-weight="700" fill="#2272B4" text-anchor="middle" xml:space="preserve">Safety</text>
<rect x="62" y="286" width="1476" height="82" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<text x="86" y="336" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Trace 3</text>
<text x="210" y="336" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">“I've already asked twice. This is really frustrating.”</text>
<rect x="900" y="310" width="85" height="34" rx="17.0" fill="#EAF2FA" stroke="#2272B4" stroke-width="1.2"/><text x="943" y="333" font-size="17" font-weight="700" fill="#2272B4" text-anchor="middle" xml:space="preserve">Safety</text>
<rect x="62" y="422" width="85" height="34" rx="17.0" fill="#EAF2FA" stroke="#2272B4" stroke-width="1.2"/><text x="105" y="445" font-size="17" font-weight="700" fill="#2272B4" text-anchor="middle" xml:space="preserve">Safety</text><text x="161" y="445" font-size="17" font-weight="600" fill="#5A6F77" text-anchor="start" xml:space="preserve">Per trace</text>
<rect x="340" y="422" width="180" height="34" rx="17.0" fill="#FFEFEC" stroke="#FF5F46" stroke-width="1.2"/><text x="430" y="445" font-size="17" font-weight="700" fill="#FF5F46" text-anchor="middle" xml:space="preserve">Multi-turn judge</text><text x="534" y="445" font-size="17" font-weight="600" fill="#5A6F77" text-anchor="start" xml:space="preserve">Per session, on Trace 1</text>
</svg>
</div>

The built-in multi-turn judges:

| Judge | What it evaluates |
|---|---|
| `ConversationCompleteness` | Whether the agent fully resolved the user's request across all turns |
| `UserFrustration` | Repeated questions, escalation language, confusion signals |
| `ConversationalSafety` | Safety violations that emerge only across multiple turns |
| `KnowledgeRetention` | Whether the agent correctly retains information from earlier in the conversation |
| `ConversationalGuidelines` | Whether responses comply with provided guidelines throughout the conversation |
| `ConversationalRoleAdherence` | Whether the agent maintains its assigned role throughout the conversation |
| `ConversationalToolCallEfficiency` | Whether tool usage across the conversation was efficient and appropriate |

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: Multi-turn judge deployment</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">from mlflow.genai.scorers import (
    ConversationCompleteness, UserFrustration,
    ConversationalSafety, ScorerSamplingConfig,
)
<span></span>
judge_model = "databricks:/workspace.agent_deployment.judge-model"
<span></span>
completeness = ConversationCompleteness(model=judge_model).register(name="conversation_completeness")
completeness = completeness.start(sampling_config=ScorerSamplingConfig(sample_rate=0.5))
<span></span>
frustration = UserFrustration(model=judge_model).register(name="user_frustration")
frustration = frustration.start(sampling_config=ScorerSamplingConfig(sample_rate=0.5))
<span></span>
# Safety checks run on every session
conv_safety = ConversationalSafety(model=judge_model).register(name="conversational_safety")
conv_safety = conv_safety.start(sampling_config=ScorerSamplingConfig(sample_rate=1.0))</div>
</div>
</details>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">A single-turn safety check might pass every response, but a user repeating the same question four times signals a broken experience only a multi-turn judge can detect. All multi-turn judges are built-in and require no external integrations. None require ground truth.</p>
</div>
</div>
</div>


### B2. Session Configuration

Sessions are identified by the `mlflow.trace.session` **metadata field**. All traces sharing the same session ID are grouped into a single conversation for multi-turn evaluation.

- Set via `mlflow.update_current_trace(session_id=..., user=...)` or equivalently via the `metadata=` dict
- Or wrap a block of code in `mlflow.tracing.context(session_id=..., user=...)` (MLflow 3.11+) so every trace inside it gets the session
- Metadata is **immutable** once logged (unlike tags which can be updated)
- A conversation is considered complete after a configurable buffer (default **5 minutes** of inactivity)
- Configure the buffer via `MLFLOW_ONLINE_SCORING_DEFAULT_SESSION_COMPLETION_BUFFER_SECONDS`

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: Session metadata configuration</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;"># Set session metadata for multi-turn grouping
# Option 1: Dedicated parameters (recommended)
mlflow.update_current_trace(
    session_id=session_id,
    user=user_id,
)
<span></span>
# Option 2: Equivalent using the metadata dict
mlflow.update_current_trace(
    metadata={
        "mlflow.trace.session": session_id,
        "mlflow.trace.user": user_id,
    }
)</div>
</div>
</details>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">Production monitoring groups traces into a conversation by their <code>mlflow.trace.session</code> value. The dedicated <code>session_id=</code> parameter sets it for you.</p>
</div>
</div>
</div>

Session metadata and multi-turn configuration (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/production-monitoring" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/production-monitoring" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/eval-monitor/production-monitoring" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

<div style="width: 100%; font-family: sans-serif;">

<div style="margin: 0 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">

  <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #FF5F46;"></div>

  <img src="https://files.training.databricks.com/binder/prod_main/deploying-and-monitoring-agents-on-databricks-en_us-1.1.0/images/20261029T133314Z/Deploying and Monitoring Agents on Databricks/Includes/images/genie-code.png" style="height: 64px; margin-bottom: 10px;">

  <div style="font-size: 15pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">
    Want to learn more about <strong>multi-turn conversation evaluation</strong>? Ask Genie Code. Click on the genie icon <img src="https://files.training.databricks.com/binder/prod_main/deploying-and-monitoring-agents-on-databricks-en_us-1.1.0/images/20261029T133314Z/Deploying and Monitoring Agents on Databricks/Includes/images/genie-icon.png" style="height: 32px; vertical-align: middle;"> and begin querying. For example, click the <strong>Copy</strong> button below and paste into <strong>Genie Code</strong>.
  </div>

  <div style="display: flex; align-items: center; gap: 10px; background: #F9F7F4; border: 1px solid #ddd; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0b2026;">
    <span id="genie-query" style="flex: 1;">How do multi-turn conversation judges work in MLflow? How do I configure sessions for conversation-level evaluation?</span>
    <button onclick="
      var text = document.getElementById('genie-query').innerText;
      var ta = document.createElement('textarea');
      ta.value = text;
      ta.style.position = 'fixed';
      ta.style.opacity = '0';
      document.body.appendChild(ta);
      ta.select();
      document.execCommand('copy');
      document.body.removeChild(ta);
      this.innerText = 'Copied!';
      var btn = this;
      setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
    " style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 4px 12px; font-size: 13pt; cursor: pointer; white-space: nowrap;">Copy</button>
  </div>

</div>

</div>

## C. Online Evaluation

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Beta</strong>
<p style="margin: 8px 0 0 0; color: #333;">Production monitoring, covered in sections C through E, is in Beta. This includes registering, starting, updating, stopping, and deleting scorers, and backfilling them on historical traces.</p>
</div>
</div>
</div>

Production monitoring runs in a background job, separate from your notebook. When an experiment stores its traces in Unity Catalog, that job reads them through a SQL warehouse. Set the warehouse on the experiment **before** you register any scorers, or the monitoring job fails:

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: Setting the monitoring SQL warehouse</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">from mlflow.tracing import set_databricks_monitoring_sql_warehouse_id
<span></span>
# Saved on the experiment as the mlflow.monitoring.sqlWarehouseId tag
set_databricks_monitoring_sql_warehouse_id(
    sql_warehouse_id="&amp;lt;warehouse-id&amp;gt;",
    experiment_id="&amp;lt;experiment-id&amp;gt;",  # optional: defaults to the active experiment
)</div>
</div>
</details>

- The `MLFLOW_TRACING_SQL_WAREHOUSE_ID` environment variable is not a substitute: it applies only to the process that sets it
- The job runs as the user who registered the first scorer. That user needs **CAN USE** on the warehouse and **CAN EDIT** on the experiment
- Reading and writing the trace tables takes **MODIFY** and **SELECT**, granted explicitly. `ALL_PRIVILEGES` is not enough
- Scorers run on serverless compute under the workspace's default serverless budget policy. If the workspace disables the default policy, registering a scorer fails with `PERMISSION_DENIED` until you set a budget policy on the experiment, in its **Details** panel or with the `mlflow.workload_creation_policy_id` experiment tag (Beta)

Set up production monitoring (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/production-monitoring" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/production-monitoring" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/eval-monitor/production-monitoring" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

### C1. Sampling Strategy

Not every scorer needs to evaluate every trace. A well-designed sampling strategy balances **coverage**, **quality signal**, and **cost**. The general principle: the higher the risk of missing an issue, the higher the sampling rate should be.

<div style="background: #fff; max-width: 1600px; margin: 0 auto; padding: 12px 0 24px;">
<svg viewBox="0 0 1600 420" width="100%" role="img" aria-label="Sampling rates by scorer type, shown on 20 traces each. Safety scorers evaluate every trace at 100%. Expensive LLM judges evaluate 5 to 10% of traces. Lightweight custom scorers use a rate that varies by use case." style="display: block; height: auto; font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;">
<defs>
<marker id="sa-ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
<filter id="sa-sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="3" stdDeviation="6" flood-color="#1B3139" flood-opacity="0.08"/></filter>
</defs>
<rect x="40" y="20" width="490" height="330" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#sa-sh)"/>
<rect x="62" y="34" width="446" height="5" rx="2.5" fill="#FF5F46"/>
<rect x="62" y="53" width="97" height="26" rx="13.0" fill="#FF5F46"/><text x="110" y="71" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">CRITICAL</text>
<text x="62" y="112" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Safety</text>
<text x="62" y="200" font-size="52" font-weight="800" fill="#FF5F46" text-anchor="start" xml:space="preserve">100%</text>
<circle cx="80" cy="250" r="14" fill="#FF5F46" stroke="#FF5F46" stroke-width="2"/>
<circle cx="124" cy="250" r="14" fill="#FF5F46" stroke="#FF5F46" stroke-width="2"/>
<circle cx="168" cy="250" r="14" fill="#FF5F46" stroke="#FF5F46" stroke-width="2"/>
<circle cx="212" cy="250" r="14" fill="#FF5F46" stroke="#FF5F46" stroke-width="2"/>
<circle cx="256" cy="250" r="14" fill="#FF5F46" stroke="#FF5F46" stroke-width="2"/>
<circle cx="300" cy="250" r="14" fill="#FF5F46" stroke="#FF5F46" stroke-width="2"/>
<circle cx="344" cy="250" r="14" fill="#FF5F46" stroke="#FF5F46" stroke-width="2"/>
<circle cx="388" cy="250" r="14" fill="#FF5F46" stroke="#FF5F46" stroke-width="2"/>
<circle cx="432" cy="250" r="14" fill="#FF5F46" stroke="#FF5F46" stroke-width="2"/>
<circle cx="476" cy="250" r="14" fill="#FF5F46" stroke="#FF5F46" stroke-width="2"/>
<circle cx="80" cy="294" r="14" fill="#FF5F46" stroke="#FF5F46" stroke-width="2"/>
<circle cx="124" cy="294" r="14" fill="#FF5F46" stroke="#FF5F46" stroke-width="2"/>
<circle cx="168" cy="294" r="14" fill="#FF5F46" stroke="#FF5F46" stroke-width="2"/>
<circle cx="212" cy="294" r="14" fill="#FF5F46" stroke="#FF5F46" stroke-width="2"/>
<circle cx="256" cy="294" r="14" fill="#FF5F46" stroke="#FF5F46" stroke-width="2"/>
<circle cx="300" cy="294" r="14" fill="#FF5F46" stroke="#FF5F46" stroke-width="2"/>
<circle cx="344" cy="294" r="14" fill="#FF5F46" stroke="#FF5F46" stroke-width="2"/>
<circle cx="388" cy="294" r="14" fill="#FF5F46" stroke="#FF5F46" stroke-width="2"/>
<circle cx="432" cy="294" r="14" fill="#FF5F46" stroke="#FF5F46" stroke-width="2"/>
<circle cx="476" cy="294" r="14" fill="#FF5F46" stroke="#FF5F46" stroke-width="2"/>
<rect x="555" y="20" width="490" height="330" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#sa-sh)"/>
<rect x="577" y="34" width="446" height="5" rx="2.5" fill="#2272B4"/>
<rect x="577" y="53" width="106" height="26" rx="13.0" fill="#2272B4"/><text x="630" y="71" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">EXPENSIVE</text>
<text x="577" y="112" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">LLM judges</text>
<text x="577" y="200" font-size="52" font-weight="800" fill="#2272B4" text-anchor="start" xml:space="preserve">5–10%</text>
<circle cx="595" cy="250" r="14" fill="#2272B4" stroke="#2272B4" stroke-width="2"/>
<circle cx="639" cy="250" r="14" fill="#2272B4" stroke="#2272B4" stroke-width="2"/>
<circle cx="683" cy="250" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="727" cy="250" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="771" cy="250" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="815" cy="250" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="859" cy="250" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="903" cy="250" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="947" cy="250" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="991" cy="250" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="595" cy="294" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="639" cy="294" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="683" cy="294" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="727" cy="294" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="771" cy="294" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="815" cy="294" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="859" cy="294" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="903" cy="294" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="947" cy="294" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="991" cy="294" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<rect x="1070" y="20" width="490" height="330" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#sa-sh)"/>
<rect x="1092" y="34" width="446" height="5" rx="2.5" fill="#00A972"/>
<rect x="1092" y="53" width="124" height="26" rx="13.0" fill="#00A972"/><text x="1154" y="71" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">LIGHTWEIGHT</text>
<text x="1092" y="112" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Custom scorers</text>
<text x="1092" y="200" font-size="52" font-weight="800" fill="#00A972" text-anchor="start" xml:space="preserve">Varies</text>
<circle cx="1110" cy="250" r="14" fill="#00A972" stroke="#00A972" stroke-width="2"/>
<circle cx="1154" cy="250" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="1198" cy="250" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="1242" cy="250" r="14" fill="#00A972" stroke="#00A972" stroke-width="2"/>
<circle cx="1286" cy="250" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="1330" cy="250" r="14" fill="#00A972" stroke="#00A972" stroke-width="2"/>
<circle cx="1374" cy="250" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="1418" cy="250" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="1462" cy="250" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="1506" cy="250" r="14" fill="#00A972" stroke="#00A972" stroke-width="2"/>
<circle cx="1110" cy="294" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="1154" cy="294" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="1198" cy="294" r="14" fill="#00A972" stroke="#00A972" stroke-width="2"/>
<circle cx="1242" cy="294" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="1286" cy="294" r="14" fill="#00A972" stroke="#00A972" stroke-width="2"/>
<circle cx="1330" cy="294" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="1374" cy="294" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="1418" cy="294" r="14" fill="#00A972" stroke="#00A972" stroke-width="2"/>
<circle cx="1462" cy="294" r="14" fill="#fff" stroke="#C6DCEA" stroke-width="2"/>
<circle cx="1506" cy="294" r="14" fill="#00A972" stroke="#00A972" stroke-width="2"/>
<circle cx="54" cy="392" r="10" fill="#5A6F77"/><text x="74" y="398" font-size="17" font-weight="600" fill="#5A6F77" text-anchor="start" xml:space="preserve">Evaluated</text>
<circle cx="204" cy="392" r="10" fill="#fff" stroke="#C6DCEA" stroke-width="2"/><text x="224" y="398" font-size="17" font-weight="600" fill="#5A6F77" text-anchor="start" xml:space="preserve">Skipped</text>
</svg>
</div>

- **Critical (100%)**: Safety and security scorers should run on every trace. The cost of missing a safety violation far outweighs evaluation cost.
- **Expensive (5-20%)**: LLM-based judges that make their own model calls are expensive per evaluation. 5% still provides statistically meaningful quality signals at scale.
- **Development (30-50%)**: While you are still tuning the agent or a judge, start higher for fast feedback, then dial back once the signal is stable.
- **Custom (variable)**: Lightweight function scorers can run at higher rates. Combine single-turn and multi-turn judges at different rates.

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: Auditing active scorers with list_scorers()</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">from mlflow.genai.scorers import list_scorers
<span></span>
# Audit all active scorers and their rates
for s in list_scorers():
    if s.sample_rate &amp;gt; 0:
        print(f"{s.name} active at {s.sample_rate}")</div>
</div>
</details>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">Maximum <strong>20 scorers</strong> per experiment. Use <code>list_scorers()</code> to audit what is running and at what rates. Change the <code>ScorerSamplingConfig(sample_rate=X)</code> of an active scorer with <code>.update()</code>, or restart a stopped scorer with <code>.start()</code>.</p>
</div>
</div>
</div>


### C2. Scorer Lifecycle

Every scorer moves through a defined set of states. Understanding these transitions is essential for managing scorers in production.

<div style="background: #fff; max-width: 1600px; margin: 0 auto; padding: 12px 0 24px;">
<svg viewBox="0 0 1600 381.1" width="100%" role="img" aria-label="Scorer lifecycle states: Created, Registered, Running, Stopped, and Deleted. register moves Created to Registered, start moves Registered to Running, stop moves Running to Stopped, and delete_scorer moves Stopped to Deleted. start returns a stopped scorer to Running, and update changes a running scorer&#x27;s sampling." style="display: block; height: auto; font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;">
<defs>
<marker id="sl-ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
<filter id="sl-sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="3" stdDeviation="6" flood-color="#1B3139" flood-opacity="0.08"/></filter>
</defs>
<rect x="32" y="120" width="200" height="149" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#sl-sh)"/>
<rect x="54" y="134" width="156" height="5" rx="2.5" fill="#5A6F77"/>
<text x="132" y="186" font-size="21" font-weight="700" fill="#0b2026" text-anchor="middle" xml:space="preserve">Created</text>
<text x="132" y="220" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="middle" xml:space="preserve">In code only</text>
<rect x="357" y="120" width="200" height="149" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#sl-sh)"/>
<rect x="379" y="134" width="156" height="5" rx="2.5" fill="#2272B4"/>
<text x="457" y="186" font-size="21" font-weight="700" fill="#0b2026" text-anchor="middle" xml:space="preserve">Registered</text>
<text x="457" y="220" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="middle" xml:space="preserve">Known to</text>
<text x="457" y="247" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="middle" xml:space="preserve">experiment</text>
<rect x="677" y="120" width="200" height="149" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#sl-sh)"/>
<rect x="699" y="134" width="156" height="5" rx="2.5" fill="#00A972"/>
<text x="777" y="186" font-size="21" font-weight="700" fill="#0b2026" text-anchor="middle" xml:space="preserve">Running</text>
<text x="777" y="220" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="middle" xml:space="preserve">Evaluating</text>
<text x="777" y="247" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="middle" xml:space="preserve">traces</text>
<rect x="997" y="120" width="200" height="149" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#sl-sh)"/>
<rect x="1019" y="134" width="156" height="5" rx="2.5" fill="#FF5F46"/>
<text x="1097" y="186" font-size="21" font-weight="700" fill="#0b2026" text-anchor="middle" xml:space="preserve">Stopped</text>
<text x="1097" y="220" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="middle" xml:space="preserve">Paused,</text>
<text x="1097" y="247" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="middle" xml:space="preserve">restartable</text>
<rect x="1382" y="120" width="200" height="149" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#sl-sh)"/>
<rect x="1404" y="134" width="156" height="5" rx="2.5" fill="#0b2026"/>
<text x="1482" y="186" font-size="21" font-weight="700" fill="#0b2026" text-anchor="middle" xml:space="preserve">Deleted</text>
<text x="1482" y="220" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="middle" xml:space="preserve">Permanent</text>
<text x="1482" y="247" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="middle" xml:space="preserve">removal</text>
<path d="M238,194.55 H349" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#sl-ar)"/>
<text x="294" y="179" font-size="17" font-weight="700" fill="#0b2026" text-anchor="middle" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">.register()</text>
<path d="M563,194.55 H669" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#sl-ar)"/>
<text x="616" y="179" font-size="17" font-weight="700" fill="#0b2026" text-anchor="middle" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">.start()</text>
<path d="M883,194.55 H989" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#sl-ar)"/>
<text x="936" y="179" font-size="17" font-weight="700" fill="#0b2026" text-anchor="middle" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">.stop()</text>
<path d="M1203,194.55 H1374" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#sl-ar)"/>
<text x="1288" y="179" font-size="17" font-weight="700" fill="#0b2026" text-anchor="middle" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">delete_scorer()</text>
<path d="M1097.0,275.1 V309.1 Q1097.0,329.1 1077.0,329.1 H797.0 Q777.0,329.1 777.0,309.1 V281.1" fill="none" stroke="#618794" stroke-width="2.4" stroke-dasharray="7 6" marker-end="url(#sl-ar)"/>
<text x="937" y="361" font-size="17" font-weight="700" fill="#0b2026" text-anchor="middle" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">.start()</text>
<path d="M737.0,114 C737.0,50 817.0,50 817.0,108" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#sl-ar)"/>
<text x="777" y="46" font-size="17" font-weight="700" fill="#0b2026" text-anchor="middle" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">.update()</text>
</svg>
</div>

- **register**: Binds the scorer to an MLflow experiment. The name must be unique within the experiment. Appears in the Scorers tab but does not evaluate traces yet.
- **start**: Activates the scorer with a `ScorerSamplingConfig`. It begins evaluating new traces at the specified sample rate.
- **stop**: Pauses evaluation. The scorer remains registered and can be restarted. Historical assessments are preserved.
- **update**: Changes the sampling configuration of an active scorer. Returns a new instance. To change a stopped scorer, restart it with `.start()`.
- **delete**: Permanently removes the scorer, whether it is active or stopped. Historical assessments remain on their traces.

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: Full scorer lifecycle (register, start, update, stop, delete)</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">from mlflow.genai.scorers import Safety, ScorerSamplingConfig, delete_scorer
<span></span>
# Full lifecycle
safety = Safety(model="databricks:/workspace.agent_deployment.judge-model").register(name="safety_v1")
safety = safety.start(sampling_config=ScorerSamplingConfig(sample_rate=1.0))
<span></span>
# Update sampling rate
safety = safety.update(sampling_config=ScorerSamplingConfig(sample_rate=0.5))
<span></span>
# Stop (preserves history)
safety = safety.stop()
<span></span>
# Restart from stopped state
safety = safety.start(sampling_config=ScorerSamplingConfig(sample_rate=1.0))
<span></span>
# Permanent removal (no need to stop first)
delete_scorer(name="safety_v1")</div>
</div>
</details>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;"><strong>Immutability pattern:</strong> Every lifecycle method (<code>.register()</code>, <code>.start()</code>, <code>.stop()</code>, <code>.update()</code>) returns a <strong>new instance</strong>. Always reassign: <code>scorer = scorer.start(...)</code>. If you call <code>scorer.stop()</code> without reassigning, the variable still points to the old (running) state.</p>
</div>
</div>
</div>


## D. Scorer Management

### D1. get_scorer()

Use `get_scorer()` to retrieve any registered scorer by name. This is essential when you need to manage a scorer that was created in a different notebook session or through the Databricks UI.

- Returns the scorer object with its current state, sample rate, and configuration
- Works with UI-created scorers and API-created scorers alike
- The returned object supports all lifecycle methods (`.start()`, `.stop()`, `.update()`)

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: Retrieving and updating a scorer by name</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">from mlflow.genai.scorers import get_scorer, ScorerSamplingConfig
<span></span>
# Fetch a scorer created in the UI or another session
my_scorer = get_scorer(name="is_english")
print(f"Name: {my_scorer.name}")
print(f"Rate: {my_scorer.sample_rate}")
<span></span>
# Update the sampling rate
updated = my_scorer.update(
    sampling_config=ScorerSamplingConfig(sample_rate=0.8)
)
<span></span>
# Original remains unchanged (immutable)
print(f"Original: {my_scorer.sample_rate}")
print(f"Updated:  {updated.sample_rate}")</div>
</div>
</details>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;"><code>get_scorer()</code> resolves against the active MLflow experiment. Make sure <code>mlflow.set_experiment()</code> points to the correct experiment before calling it.</p>
</div>
</div>
</div>


### D2. Stopping and Deleting

**Stopping** pauses a running scorer. It remains registered and can be restarted at any time. The background Trace Metrics Computation Job continues running until all scorers in the experiment are stopped.

**Deleting** permanently removes a scorer from the experiment. Historical assessments produced by the scorer are preserved on their traces.

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: Stopping and restarting individual scorers</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">from mlflow.genai.scorers import get_scorer, list_scorers, delete_scorer, ScorerSamplingConfig
<span></span>
# Stop a scorer (always reassign)
safety = safety.stop()
<span></span>
# Stop a scorer fetched by name
english_judge = get_scorer(name="is_english")
english_judge = english_judge.stop()
<span></span>
# Resume later
safety = safety.start(
    sampling_config=ScorerSamplingConfig(sample_rate=1.0)
)</div>
</div>
</details>

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: Bulk cleanup</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;"># Bulk cleanup: delete all scorers (stopping first is optional)
for s in list_scorers():
    delete_scorer(name=s.name)
<span></span>
# Verify
print(list_scorers())  # []</div>
</div>
</details>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;"><strong>.stop() vs. delete_scorer():</strong> Stopping pauses evaluation but keeps the scorer registered (it can be restarted). Deleting permanently removes it, whether it is running or stopped. Historical assessments are preserved in both cases.</p>
</div>
</div>
</div>


## E. Backfill and Trace Storage

### E1. Metric Backfill

**Metric backfill** applies scorers to traces logged before the scorer existed. This lets you retroactively evaluate historical traces with new quality checks without reprocessing the original requests.

<div style="background: #fff; max-width: 1600px; margin: 0 auto; padding: 12px 0 24px;">
<svg viewBox="0 0 1600 314" width="100%" role="img" aria-label="Metric backfill: historical traces logged before a scorer existed go to a backfill job started with backfill_scorers, which writes assessments back onto the same traces." style="display: block; height: auto; font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;">
<defs>
<marker id="baba-ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
<filter id="baba-sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="3" stdDeviation="6" flood-color="#1B3139" flood-opacity="0.08"/></filter>
</defs>
<rect x="40" y="20" width="97" height="26" rx="13.0" fill="#FF5F46"/><text x="88" y="38" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">BACKFILL</text>
<rect x="60" y="64" width="450" height="130" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#baba-sh)"/>
<rect x="82" y="78" width="406" height="5" rx="2.5" fill="#FF5F46"/>
<text x="82" y="130" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Historical traces</text>
<text x="82" y="164" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">Logged before the scorer</text>
<rect x="590" y="64" width="450" height="130" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#baba-sh)"/>
<rect x="612" y="78" width="406" height="5" rx="2.5" fill="#FF5F46"/>
<text x="612" y="130" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Backfill job</text>
<text x="612" y="164" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">backfill_scorers()</text>
<path d="M516,129 H582" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#baba-ar)"/>
<rect x="1120" y="64" width="450" height="130" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#baba-sh)"/>
<rect x="1142" y="78" width="406" height="5" rx="2.5" fill="#FF5F46"/>
<text x="1142" y="130" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Assessments</text>
<text x="1142" y="164" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">On the same traces</text>
<path d="M1046,129 H1112" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#baba-ar)"/>
<path d="M1345.0,200 V224 Q1345.0,240 1329.0,240 H301.0 Q285.0,240 285.0,224 V206" fill="none" stroke="#618794" stroke-width="2.4" stroke-dasharray="7 6" marker-end="url(#baba-ar)"/>
</svg>
</div>

- `backfill_scorers` is from `databricks.agents.scorers`
- Creates a **[experiment_id] Metric Backfill Computation Job** in Jobs and Pipelines
- Scorers must be registered and started before you can backfill them
- Pass a list of scorer name strings to use their current sample rates, or use `BackfillScorerConfig` for custom rates
- With no `start_time` or `end_time`, the backfill covers only the last seven days. Pass a `start_time` to reach older traces
- With only an `end_time`, the backfill starts one day before it
- Returns a `job_id` for tracking progress

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: Two ways to call backfill_scorers</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">from databricks.agents.scorers import backfill_scorers, BackfillScorerConfig
from mlflow.genai.scorers import get_scorer
from datetime import datetime, timedelta
<span></span>
# Simple backfill using registered sample rates
job_id = backfill_scorers(
    scorers=["safety_check", "response_length"]
)
<span></span>
# Or: custom sample rates and a time range, using the registered scorers
safety_judge = get_scorer(name="safety_check")
response_length = get_scorer(name="response_length")
job_id = backfill_scorers(
    experiment_id="YOUR_EXPERIMENT_ID",
    scorers=[
        BackfillScorerConfig(scorer=safety_judge, sample_rate=1.0),
        BackfillScorerConfig(scorer=response_length, sample_rate=0.9),
    ],
    start_time=datetime.now() - timedelta(days=7),
    end_time=datetime.now(),
)</div>
</div>
</details>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;"><strong>Best practices:</strong> Test with a narrow time range first. Use lower sample rates for expensive judges during large backfills. Verify scorer names with <code>list_scorers()</code> before starting. Track progress via the <code>job_id</code> in the Jobs UI.</p>
</div>
</div>
</div>


### E2. Traces in Unity Catalog

Databricks recommends storing production traces in Unity Catalog. The experiment is still where you view traces in the UI, but the traces themselves are written to Delta tables in your schema, in OpenTelemetry format. This replaces the older trace archival pattern, where a separate Trace Archive Job (`enable_databricks_trace_archival`) copied traces from the experiment into a Delta table.

| | Traces in Unity Catalog | Traces in the experiment |
|---|---|---|
| **Storage limit** | Unlimited | 100,000 traces per experiment |
| **Query options** | MLflow UI and SDK, plus SQL, AI/BI dashboards, Genie Code, and Spark | MLflow UI and SDK |
| **Governance** | Unity Catalog schema and table permissions | Experiment permissions |
| **Format** | OpenTelemetry, readable by other OTel tools | MLflow only |

- Bind the location with `mlflow.set_experiment(..., trace_location=UnityCatalog(...))`. An experiment can only be bound when it is created
- MLflow creates four tables (`&lt;prefix&gt;_otel_spans`, `_otel_logs`, `_otel_annotations`, and `_otel_metrics`) and two views over them: `&lt;prefix&gt;_trace_unified`, with span data, and `&lt;prefix&gt;_trace_metadata`, without it
- Requires MLflow 3.14 or later and a SQL warehouse, set with `MLFLOW_TRACING_SQL_WAREHOUSE_ID`
- Assessments from monitoring and backfill are stored with the traces, so you can query them with SQL

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: Creating an experiment that stores traces in Unity Catalog</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">import os
import mlflow
from mlflow.entities.trace_location import UnityCatalog
<span></span>
# MLflow writes and searches traces in Unity Catalog through a SQL warehouse
os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = "&amp;lt;warehouse-id&amp;gt;"
<span></span>
# Bind the trace location when the experiment is created
mlflow.set_experiment(
    experiment_name="/Workspace/Users/&amp;lt;user&amp;gt;/my-agent",
    trace_location=UnityCatalog(
        catalog_name="my_catalog",
        schema_name="my_schema",
        table_prefix="my_agent",
    ),
)</div>
</div>
</details>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">You can't delete individual traces stored in Unity Catalog from the MLflow UI or API. To remove traces, delete rows from the trace tables with SQL.</p>
</div>
</div>
</div>

Store traces in Unity Catalog (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/tracing/trace-unity-catalog" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/tracing/trace-unity-catalog" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
